#Load Dataset


In [1]:
from datasets import load_dataset
import numpy as np

dataset = load_dataset("Kibalama/wlasl-upper-arm-pose-landmarks")

train_data = dataset["train"]

print(dataset)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/9.60M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1000 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['video_id', 'poses', 'label'],
        num_rows: 1000
    })
})


# Extract X (landmarks) + y (labels)

In [2]:
X = []
y = []

for sample in train_data:
    X.append(np.array(sample["poses"]))  # (frames, 7, 3)
    y.append(sample["label"])

# Fix NaN values

In [3]:
clean_X = []

for seq in X:
    seq = np.array(seq)
    seq = np.nan_to_num(seq, nan=0.0)
    clean_X.append(seq)

# Fix sequence length


In [4]:
MAX_LEN = 60

def pad_sequence(seq):
    if len(seq) > MAX_LEN:
        return seq[:MAX_LEN]

    elif len(seq) < MAX_LEN:
        pad = np.zeros((MAX_LEN - len(seq), seq.shape[1], seq.shape[2]))
        seq = np.concatenate([seq, pad], axis=0)

    return seq

In [5]:
X = np.array([pad_sequence(x) for x in clean_X])

# Normalize data

In [6]:
X = (X - np.mean(X)) / (np.std(X) + 1e-8)

# Encode labels

In [7]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
y_encoded = le.fit_transform(y)

# Train / Test split


In [8]:
from sklearn.model_selection import train_test_split

X_train, X_val, y_train, y_val = train_test_split(
    X,
    y_encoded,
    test_size=0.2,
    random_state=42
)

# FINAL DATA CHECK

In [9]:
print("X shape:", X.shape)
print("y shape:", y_encoded.shape)

X shape: (1000, 60, 7, 3)
y shape: (1000,)


In [10]:
print(set([x.shape[0] for x in X]))

{60}


In [11]:
print("NaN:", np.isnan(X).sum())
print("Inf:", np.isinf(X).sum())

NaN: 0
Inf: 0


In [12]:
print("min:", np.min(X))
print("max:", np.max(X))

min: -3.7391180519004314
max: 2.7472718441837007


In [13]:
print("num classes:", len(set(y_encoded)))

num classes: 184


In [14]:
print("sample shape:", X[0].shape)
print("sample data:", X[0][:2])

sample shape: (60, 7, 3)
sample data: [[[ 0.75596524 -2.32246791 -0.85267879]
  [ 1.71976522 -1.34722538 -0.19204353]
  [ 0.14654481 -1.46402213  0.08077046]
  [ 1.73167153 -0.25870795 -0.0974143 ]
  [-0.12395078 -0.37102445  0.33823602]
  [ 1.32983248  0.59445287 -0.48200739]
  [ 0.01185803  0.46996519  0.02907435]]

 [[ 0.75596062 -2.32233765 -0.85700615]
  [ 1.71266429 -1.33393841 -0.19100379]
  [ 0.14656051 -1.45796095  0.04435282]
  [ 1.74069862 -0.2298998  -0.09741627]
  [-0.13583636 -0.34851428  0.26973581]
  [ 1.39825964  0.61630065 -0.48126015]
  [ 0.00298764  0.47696686 -0.01410681]]]
